In [1]:
import random
import json
from datetime import datetime, timedelta

PAGE_NAMES = ['홈', '주문완료', '마이페이지', '주문결제', '로그인', '상품목록', '장바구니', '회원가입']
END_DATE   = datetime(2026, 6, 17, 23, 59, 59)
START_DATE = datetime(2026, 1, 1, 0, 0, 0)
ROW_COUNT  = 500  # 생성할 로그 수

def random_datetime(start, end):
    delta = end - start
    random_seconds = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_seconds)

def format_kst(dt):
    return dt.strftime('%Y-%m-%dT%H:%M:%S.000+09:00')

rows = []
for _ in range(ROW_COUNT):
    history_ts = random_datetime(START_DATE, END_DATE)
    event_ts   = history_ts + timedelta(seconds=1)
    page_name  = random.choice(PAGE_NAMES)
    dwell_time = random.randint(1, 30)
    user_id    = f'user{random.randint(1, 100):04d}'

    json_log = json.dumps({
        'event_name':      'page_view',
        'pageName':        page_name,
        'dwellTime':       dwell_time,
        'user_id':         user_id,
        'event_timestamp': format_kst(event_ts)
    }, ensure_ascii=False)

    rows.append((history_ts, json_log))

# SQL 생성
lines = []
lines.append("INSERT INTO first_save_history (history_timestamp, json_log) VALUES")
values = []
for history_ts, json_log in rows:
    ts_str   = history_ts.strftime('%Y-%m-%d %H:%M:%S.%f')
    escaped  = json_log.replace("'", "''")
    values.append(f"  ('{ts_str}', '{escaped}')")

lines.append(',\n'.join(values) + ';')
sql = '\n'.join(lines)

with open('page_view_logs.sql', 'w', encoding='utf-8') as f:
    f.write(sql)

print(f'✅ {ROW_COUNT}개 page_view 로그 SQL 생성 완료 → page_view_logs.sql')
print(sql[:500])

✅ 500개 page_view 로그 SQL 생성 완료 → page_view_logs.sql
INSERT INTO first_save_history (history_timestamp, json_log) VALUES
  ('2026-04-16 03:17:46.000000', '{"event_name": "page_view", "pageName": "홈", "dwellTime": 20, "user_id": "user0099", "event_timestamp": "2026-04-16T03:17:47.000+09:00"}'),
  ('2026-04-05 17:54:12.000000', '{"event_name": "page_view", "pageName": "로그인", "dwellTime": 27, "user_id": "user0002", "event_timestamp": "2026-04-05T17:54:13.000+09:00"}'),
  ('2026-01-15 08:19:05.000000', '{"event_name": "page_view", "pageName": "장바구니", 
